## Run GridSearch for Surface Coverage

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
import pylupnt as pnt

In [ ]:
import os

os.makedirs("data/gridsearch_coverage", exist_ok=True)
os.makedirs("figs/coverage", exist_ok=True)
sma_24hr = (pnt.GM_MOON * (24 * 3600) ** 2 / (4 * np.pi**2)) ** (1 / 3)
print("24-hour orbit sma: {:.1f} km".format(sma_24hr / 1000))

In [ ]:
et0 = pnt.convert_time(pnt.gregorian_to_time(2030, 1, 1, 12, 0, 0), pnt.UTC, pnt.TAI)

# constants
n_sma = 13
n_incs = int((70 - 40) / 2.5) + 1  # 12 increments of 2.5 deg

# results
results = np.ones((n_sma, n_incs, 3)) * np.nan  # RMS, 95%, 99.7% (m)

# initial conditions
smas = np.linspace(4000, 16000, n_sma) * 1e3  # [km] -> [m]
incs = np.deg2rad(np.linspace(40, 70, n_incs, endpoint=True))  # [deg] -> [rad]

print("sma (km):", smas / 1e3)
print("inc (deg):", np.rad2deg(incs))

## surface points

In [ ]:
from src.constellation_design import fibonacci_sphere
from src.postprocess import plot_user_positions

n_user = 200  # number of users to simulate
x_user = (
    fibonacci_sphere(n_user) * pnt.R_MOON
)  # [km] user positions on the surface of the Moon
# plot_user_positions(
#     x_user,
#     lat_range=(-90, 90),
#     marker_size=2,
#     camera_view=(-40, 20, 2.5),
#     fignaeme=f"figs/coverage/user_positions_{n_user}.pdf",
# )

# latitude band
plot_user_positions(
    x_user,
    lat_range=(-90, -75),
    marker_size=2,
    camera_view=(-30, -30, 2.5),
    fignaeme=f"figs/coverage/user_positions_latband_{n_user}.pdf",
)

In [ ]:
import matplotlib.pyplot as plt
import os
from src.postprocess import plot_coverage_map

## 1. Global Coverage Analysis

In [ ]:
# pattern (plane, sats per plane)

# walker patterns to test
walker_patterns = []
# 6 sats
walker_patterns += [(2, 3), (3, 2)]
# 8 sats
walker_patterns += [(2, 4), (4, 2)]
# 9 sats
walker_patterns += [(3, 3)]
# 10 sats
walker_patterns += [(2, 5), (5, 2)]
# 12 sats
walker_patterns += [(3, 4), (4, 3), (2, 6), (6, 2)]
# 14 sats
walker_patterns += [(2, 7)]
# 16 sats
walker_patterns += [(4, 4)]
# 18 sats
walker_patterns += [(6, 3), (3, 6)]
# 20 sats
walker_patterns += [(4, 5), (5, 4)]
# 21 sats
walker_patterns += [(7, 3), (3, 7)]
# 24 sats
walker_patterns += [(8, 3), (3, 8), (4, 6), (6, 4)]
# 25 sats
walker_patterns += [(5, 5)]

### 1-1: Non-hybrid Case 
First, we consider a case where we consider a simple hybrid constellation

In [ ]:
from src.gridsearch import gridsearch_coverage
import os

recalc = False  # if True, recalculate the gridsearch, otherwise load existing results
datadir_base = "data/gridsearch_coverage"
if not os.path.exists(datadir_base):
    os.makedirs(datadir_base)

# params
dt_sim = 300.0  # [s] simulation duration (every 5 minutes)
n_user = 200  # number of users to simulate

datadir = os.path.join(
    datadir_base, "nuser_{0:d}_dt_{1:.0f}_hybrid_0".format(n_user, dt_sim)
)
if not os.path.exists(datadir):
    os.makedirs(datadir)

for wi, walker_pattern in enumerate(walker_patterns):

    print("-------------------------------------")
    print(wi, "/", len(walker_patterns), "  pattern:", walker_pattern)
    print("-------------------------------------")

    filename = os.path.join(
        datadir, "walker_{0:d}_{1:d}.npy".format(walker_pattern[0], walker_pattern[1])
    )

    if os.path.exists(filename) and not recalc:
        result = np.load(filename)
        print("Loaded existing results from", filename)
    else:
        result = gridsearch_coverage(
            et0,
            smas,
            incs,
            walker_pattern=walker_pattern,
            min_elev_deg=5.0,
            n_user=n_user,
            dt_sim=dt_sim,
            pole_lat=-75.0,
            use_hybrid=False,
            parallel=True,
            max_workers=10,
        )

    # save results
    np.save(filename, result)

### Pole coverage

In [ ]:
walker_patterns_polar_plot = [(2, 3), (2, 4), (4, 2), (2, 5), (3, 4), (4, 3)]
smas = np.linspace(4000, 16000, n_sma) * 1e3  # [km] -> [m]
incs = np.deg2rad(np.linspace(40, 70, n_incs, endpoint=True))  # [deg] -> [rad]

fig = plot_coverage_map(
    datadir,
    walker_patterns_polar_plot,
    smas,
    incs,
    n_cols=3,
    plot_pole=True,
    with_text=False,
)
fig.tight_layout()
plt.savefig("figs/coverage/coverage_gridsearch_pole.pdf", dpi=300)
fig.show()

### Global Coverage

In [ ]:
walker_patterns_polar_plot = [(2, 3), (4, 2), (2, 5), (3, 4), (4, 3), (2, 6)]
smas = np.linspace(4000, 16000, n_sma) * 1e3  # [km] -> [m]
incs = np.deg2rad(np.linspace(40, 70, n_incs, endpoint=True))  # [deg] -> [rad]

fig = plot_coverage_map(
    datadir,
    walker_patterns_polar_plot,
    smas,
    incs,
    n_cols=3,
    plot_pole=True,
    with_text=False,
)
fig.tight_layout()
plt.savefig("figs/coverage/coverage_gridsearch_global.pdf", dpi=300)
fig.show()

### 1-2: Hybrid Constellation
Next, we consider a hybrid constellation case, where we combine north (w=-90) and south (w=90) ELFOs.

In [ ]:
walker_patterns_hybrid = []

for walker in walker_patterns:
    nsat = walker[0] * walker[1]
    if (
        nsat <= 12
    ):  # limit to 12 satellites for hybrid constellation since the number will be doubled
        walker_patterns_hybrid.append(walker)

print("Walker patterns for hybrid constellation:", walker_patterns_hybrid)

In [ ]:
recalc = False  # if True, recalculate the gridsearch, otherwise load existing results
datadir_base = "data/gridsearch_coverage"
if not os.path.exists(datadir_base):
    os.makedirs(datadir_base)

# params
dt_sim = 300.0  # [s] simulation duration
n_user = 200  # number of users to simulate

datadir_h = os.path.join(
    datadir_base, "nuser_{0:d}_dt_{1:.0f}_hybrid_1".format(n_user, dt_sim)
)
if not os.path.exists(datadir_h):
    os.makedirs(datadir_h)

for wi, walker_pattern in enumerate(walker_patterns_hybrid):

    print("-------------------------------------")
    print(wi, "/", len(walker_patterns_hybrid), "  pattern:", walker_pattern)
    print("-------------------------------------")

    filename = os.path.join(
        datadir_h, "walker_{0:d}_{1:d}.npy".format(walker_pattern[0], walker_pattern[1])
    )

    if os.path.exists(filename) and not recalc:
        result = np.load(filename)
        print("Loaded existing results from", filename)
    else:
        result = gridsearch_coverage(
            et0,
            smas,
            incs,
            walker_pattern=walker_pattern,
            min_elev_deg=5.0,
            n_user=n_user,
            dt_sim=dt_sim,
            pole_lat=-75.0,
            use_hybrid=True,
            parallel=True,
            max_workers=10,
        )

    # save results
    np.save(filename, result)

In [ ]:
# global coverage
walker_patterns_hybrid_plot = [(2, 3), (2, 4), (3, 3), (2, 5), (3, 4), (2, 6)]
smas = np.linspace(4000, 16000, n_sma) * 1e3  # [km] -> [m]
incs = np.deg2rad(np.linspace(40, 70, n_incs, endpoint=True))  # [deg] -> [rad]

fig = plot_coverage_map(
    datadir_h,
    walker_patterns_hybrid_plot,
    smas,
    incs,
    plot_pole=False,
    hybrid=True,
    n_cols=3,
)
plt.savefig("figs/coverage/coverage_gridsearch_hybrid_global.pdf", dpi=300)
fig.show()

#

## 2. DOP Analysis

In [ ]:
from src.constellation_design import compute_enu_matrices, fibonacci_sphere

# precompute user positions and ENU matrices for DOP calculation
# Users & ENU matrices (shared, read-only)
lat_users_vec = np.linspace(-np.pi / 2, np.pi / 2, 45)
lon_users_vec = np.linspace(-np.pi, np.pi, 90)
lat_grid, lon_grid = np.meshgrid(lat_users_vec, lon_users_vec)
x_user_grid = np.zeros((lat_grid.size, 3))
x_user_grid[:, 0] = pnt.R_MOON * np.cos(lat_grid.ravel()) * np.cos(lon_grid.ravel())
x_user_grid[:, 1] = pnt.R_MOON * np.cos(lat_grid.ravel()) * np.sin(lon_grid.ravel())
x_user_grid[:, 2] = pnt.R_MOON * np.sin(lat_grid.ravel())
#
enu_mats_grid = compute_enu_matrices(x_user_grid)  # (n_users, 3, 3)

plot_user_positions(x_user_grid, lat_range=(-90, 90), marker_size=2)

## 2-1. DOP Map Plots
moved to `run_dop_plots.py`

## 2.2. DOP GridSearch

### 2-2-1. Pole Coverage (ELFO)

In [ ]:
from src.gridsearch import gridsearch_dop_polar
import os

os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

incs = np.deg2rad(np.linspace(40, 62.5, 10))
n_user = 200
dt_sim = 300.0  # [s] simulation duration (every 5 minutes)

print(np.rad2deg(incs))
print(incs.shape)

a_hr = 24  # hours
sma = (pnt.GM_MOON * (a_hr * 3600) ** 2 / (4 * np.pi**2)) ** (1 / 3)
print("{:d}-hour orbit sma: {:.1f} km".format(a_hr, sma / 1000))

In [ ]:
sma_polar, incs, walker_patterns_polar, results_polar = gridsearch_dop_polar(
    sma, incs, n_user, et0, dt_sim, recompute=False, sat_fault=False
)

In [ ]:
from src.postprocess import plot_dop_gridsearch_polar

walker_patterns_plot_polar = [(4, 2), (2, 5), (5, 2), (3, 4), (4, 3), (2, 6)]
plot_dop_gridsearch_polar(sma_polar, incs, walker_patterns_plot_polar, results_polar)

### 2-2-2. Global Coverage and PDOP (Hybrid)
Analyze how the 95% PDOP changes for different families and inclinations

In [ ]:
from src.gridsearch import gridsearch_dop_hybrid

incs = np.deg2rad(np.linspace(40, 62.5, 10))
n_user = 200

print(incs.shape)

a_hr = 24  # hours
sma = (pnt.GM_MOON * (a_hr * 3600) ** 2 / (4 * np.pi**2)) ** (1 / 3)
print("{:d}-hour orbit sma: {:.1f} km".format(a_hr, sma))

sma_hybrid, incs, walker_patterns_hybrid, results_hybrid = gridsearch_dop_hybrid(
    sma, incs, n_user, et0, dt_sim, recompute=False, sat_fault=False
)

In [ ]:
from src.postprocess import plot_dop_gridsearch_hybrid

walker_patterns_hybrid_plot = [(4, 2), (3, 3), (2, 5), (3, 4), (4, 3), (2, 6)]
plot_dop_gridsearch_hybrid(
    sma_hybrid, incs, walker_patterns_hybrid_plot, results_hybrid
)

### 2-2-3. Global Coverage (Circular)
Analyze the 95% dop values and coverage for different families and SMAs

In [ ]:
from src.gridsearch import gridsearch_dop_circular

smas = np.linspace(5000, 16000, 23) * 1e3  # [m]
n_user = 200

smas, inc, walker_patterns_circular, results_circ = gridsearch_dop_circular(
    smas, n_user, et0, dt_sim, recompute=False, sat_fault=False
)

In [ ]:
from src.postprocess import plot_dop_gridsearch_circular

walker_patterns_circular_plot = [(4, 4), (3, 6), (4, 5), (3, 7), (4, 6), (3, 8)]
plot_dop_gridsearch_circular(smas, inc, walker_patterns_circular_plot, results_circ)

## 3. Robustness Analysis

### 3.1 Pole Coverage (Fault)

In [ ]:
# pole coverage
incs = np.deg2rad(np.linspace(40, 62.5, 10))
n_user = 200

print(incs.shape)

a_hr = 24
sma = (pnt.GM_MOON * (a_hr * 3600) ** 2 / (4 * np.pi**2)) ** (1 / 3)
print("{:d}-hour orbit sma: {:.1f} km".format(a_hr, sma))

sma_polar, incs, walker_patterns_polar, results_polar_fault = gridsearch_dop_polar(
    sma, incs, n_user, et0, dt_sim, recompute=False, sat_fault=True
)

In [ ]:
plot_dop_gridsearch_polar(
    sma_polar, incs, walker_patterns_polar_plot, results_polar_fault, is_fault=True
)

### 3.2 Global Coverage with Hybrid (Fault)

In [ ]:
# hybrid constellation
incs = np.deg2rad(np.linspace(40, 62.5, 10))
n_user = 200

print(incs.shape)

a_hr = 24
sma = (pnt.GM_MOON * (a_hr * 3600) ** 2 / (4 * np.pi**2)) ** (1 / 3)
print("{:d}-hour orbit sma: {:.1f} km".format(a_hr, sma))

sma_hybrid, incs, walker_patterns_hybrid, results_hybrid_fault = gridsearch_dop_hybrid(
    sma, incs, n_user, et0, dt_sim, recompute=False, sat_fault=True
)

In [ ]:
plot_dop_gridsearch_hybrid(
    sma_hybrid, incs, walker_patterns_hybrid, results_hybrid_fault, is_fault=True
)

### 3.3 Global Coverage with Circular (Fault)

In [ ]:
smas_circ = np.linspace(5000, 16000, 23) * 1e3  # [m]
n_user = 200

smas_circ, inc, walker_patterns_circular, results_circ_fault = gridsearch_dop_circular(
    smas_circ, n_user, et0, dt_sim, recompute=False, sat_fault=True
)

In [ ]:
plot_dop_gridsearch_circular(
    smas_circ, inc, walker_patterns_circular_plot, results_circ_fault, is_fault=True
)

In [ ]:
def print_perfomance_degrade(
    results, results_fault, incs, walker_patterns, is_hybrid=False, is_polar=False
):
    print("Performance degrade due to 1 satellite fault:")

    # print header
    #   nplane   nsat  | cov_99  cov_99_fault  degrade_cov_99  |  pdop6  pdop6_fault  degrade_pdop6  |  pdop3  pdop3_fault  degrade_pdop3
    print(
        "  nplane  nsat  |  inc  | cov_99  cov_99_fault  degrade_cov_99  |  pdop6  pdop6_fault  degrade_pdop6  |  pdop3  pdop3_fault  degrade_pdop3"
    )
    print(
        "------------------------------------------------------------------------------------------------------------------------------"
    )

    for k, walker_pattern in enumerate(walker_patterns):
        nsat = walker_pattern[0] * walker_pattern[1]
        nplane = walker_pattern[0]

        if is_hybrid:
            nsat = nsat * 2  # hybrid constellation has double the number of satellites
            nplane = nplane * 2

        idx_cov = 0
        idx_dop6 = 2

        if is_polar:
            idx_dop3 = 3
        else:
            idx_dop3 = 7

        if is_hybrid == False and is_polar == False:
            idxi = np.where(smas_circ < sma_24hr)[0][
                -1
            ]  # find the largest sma that is less than the 24-hour orbit sma
            inc_best = np.deg2rad(39.28)  # circular => critical inclination
        else:
            idxi = np.argmax(
                results[walker_pattern][:, idx_dop6]
            )  # find the inclination with the best PDOP under 6 performance
            inc_best = incs[idxi]

        cov_99 = results[walker_pattern][idxi, idx_cov]
        cov_99_fault = results_fault[walker_pattern][idxi, idx_cov]
        pdop6 = results[walker_pattern][idxi, idx_dop6]
        pdop6_fault = results_fault[walker_pattern][idxi, idx_dop6]
        pdop3 = results[walker_pattern][idxi, idx_dop3]
        pdop3_fault = results_fault[walker_pattern][idxi, idx_dop3]

        # print Table
        # header
        #   nplane   nsat  | cov_99  cov_99_fault  degrade_cov_99  |  pdop6  pdop6_fault  degrade_pdop6  |  pdop3  pdop3_fault  degrade_pdop3
        print(
            "  {0:2d}   &  {1:2d}   & {2:4.2f} &  {3:6.2f} &  {4:6.2f} &      {5:6.2f}       &    {6:6.2f} & {7:6.2f} & {8:6.2f}    &    {9:6.2f} & {10:6.2f} & {11:6.2f} \\\ ".format(
                nplane,
                nsat,
                np.rad2deg(inc_best),
                100 * cov_99,
                100 * cov_99_fault,
                100 * (cov_99 - cov_99_fault),
                100 * pdop6,
                100 * pdop6_fault,
                100 * (pdop6 - pdop6_fault),
                100 * pdop3,
                100 * pdop3_fault,
                100 * (pdop3 - pdop3_fault),
            )
        )


print_perfomance_degrade(
    results_polar,
    results_polar_fault,
    incs,
    walker_patterns_polar,
    is_hybrid=False,
    is_polar=True,
)

In [ ]:
print_perfomance_degrade(
    results_hybrid,
    results_hybrid_fault,
    incs,
    walker_patterns_hybrid,
    is_hybrid=True,
    is_polar=False,
)

In [ ]:
print_perfomance_degrade(
    results_circ,
    results_circ_fault,
    incs,
    walker_patterns_circular,
    is_hybrid=False,
    is_polar=False,
)